## RAG 入门——搭建 OA 工单知识库问答系统
把自己的文档（OA 工单、制度文件、FAQ）存入向量数据库，并让模型基于这些文档回答问题。  
RAG = Retrieval-Augmented Generation（检索增强生成）

#### 没有RAG
用户问："离职流程是什么？"  
&emsp;&emsp;&emsp;&emsp;↓  
模型回答："离职流程一般包括提交申请、上级审批、HR 办理等步骤..."  
&emsp;&emsp;&emsp;&emsp;↓  
⚠️ 模型可能胡说，因为它没见过你们公司的真实流程  
#### 有RAG
用户问："离职流程是什么？"  
&emsp;&emsp;&emsp;&emsp;↓  
① 把问题转换成向量，去向量库里找最相似的公司文档  
② 找到："《员工离职管理制度》第 3 条：员工需在 OA 提交离职申请..."  
③ 把找到的文档片段 + 原始问题一起发给模型  
&emsp;&emsp;&emsp;&emsp;↓  
模型回答："根据公司制度，离职流程如下：1. 在 OA 提交离职申请..."  
&emsp;&emsp;&emsp;&emsp;↓  
✅ 回答基于真实文档，准确且可追溯  

#### RAG 三要素：
文档：你的 OA 工单、制度文件、FAQ  
向量库：存文档的地方，能按语义相似度检索  
大模型：基于检索到的文档生成回答  

In [1]:
# chromadb：轻量级向量数据库，不需要额外启动服务
# sentence-transformers：把文本转换成向量（embedding）
pip install chromadb sentence-transformers

In [1]:
# 设置 Hugging Face 镜像为国内的 hf-mirror.com
# 必须把环境变量放在最最最前面，在任何 chromadb / sentence_transformers import 之前
# 常见坑：
# 在 Jupyter / VS Code 里之前已经 import 过 sentence_transformers，环境变量就不会再生效
# 解决：重启 Python 内核 / 终端，再跑上面代码
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 报错：网络环境无法访问 Hugging Face 的服务器（被墙了）

import chromadb
print(chromadb.__version__)

from sentence_transformers import SentenceTransformer
print("安装成功")

1.5.9


D:\software\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


安装成功


## 第 1 步：准备你的文档数据
新建 documents.py，存放你的知识库数据

In [2]:
# OA 工单知识库示例数据
oa_tickets = [
    {
        "id": "TK-001",
        "title": "离职流程咨询",
        "content": "员工离职需要在 OA 系统提交离职申请，经部门主管审批、HR 审核、财务清算后，由 HR 办理离职手续。整个流程预计 5-7 个工作日。"
    },
    {
        "id": "TK-002",
        "title": "加班审批流程",
        "content": "加班需提前在 OA 提交加班申请，注明加班日期、时长、事由。经部门主管审批后生效。紧急情况可在加班后 24 小时内补交申请。"
    },
    {
        "id": "TK-003",
        "title": "报销单审批时限",
        "content": "报销单提交后，部门主管需在 3 个工作日内完成审批，财务部门在 5 个工作日内完成审核付款。超时未审批的系统自动提醒。"
    },
    {
        "id": "TK-004",
        "title": "ERP 系统账号申请",
        "content": "新员工入职后，由部门文员在 OA 提交 ERP 账号申请，IT 部门在 1 个工作日内开通。账号权限按岗位职责分配，不得私自授权。"
    },
    {
        "id": "TK-005",
        "title": "BOM 变更流程",
        "content": "BOM 变更需由工程师在 PLM 系统提交变更申请，经项目经理审批后，通知物料管理员更新物料清单，涉及产线调整的需同步通知生产计划员。"
    },
    {
        "id": "TK-006",
        "title": "固定资产领用",
        "content": "领用固定资产（电脑、显示器等）需在 OA 提交资产领用申请，经部门主管审批后到行政部领取。贵重设备需 IT 部门登记备案。"
    },
    {
        "id": "TK-007",
        "title": "请假类型及天数",
        "content": "年假：工龄满 1 年享 5 天，满 10 年享 10 天。病假：需提供医院证明，全年累计不超过 30 天。事假：全年累计不超过 15 天。婚假：13 天。产假：158 天。"
    },
    {
        "id": "TK-008",
        "title": "会议室预约规则",
        "content": "会议室通过 OA 系统预约，单次最长预约 4 小时。如需取消，请在预约时间前 2 小时取消。月度累计爽约 3 次以上，暂停预约权限一个月。"
    },
]

# 制度文档示例
policy_docs = [
    {
        "id": "POL-001",
        "title": "信息安全管理制度",
        "content": "员工不得将公司数据外泄，不得使用未经授权的软件。办公电脑必须安装公司指定的杀毒软件，每月至少更新一次密码。违反规定者视情节严重给予警告至开除处分。"
    },
    {
        "id": "POL-002",
        "title": "考勤管理制度",
        "content": "工作时间：周一至周五 9:00-18:00，午休 12:00-13:30。迟到 30 分钟内扣 50 元，超过 30 分钟按半天事假处理。每月迟到累计 5 次以上，取消当月全勤奖。"
    },
]

## 第 2 步：把文档存入 Chroma 向量库
新建 build_knowledge_base.py：

In [5]:
# 设置 Hugging Face 镜像为国内的 hf-mirror.com
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 报错：网络环境无法访问 Hugging Face 的服务器（被墙了）


import chromadb
from chromadb.utils import embedding_functions

# ---------- 1. 初始化 Chroma 客户端 ----------
# 持久化存储到本地文件夹
client = chromadb.PersistentClient(path="./chroma_db")

# 使用 sentence-transformers 作为 embedding 模型
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"  # 轻量级模型，英文效果好；中文也有替代品
                                                       )
# sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
#     model_name="paraphrase-multilingual-MiniLM-L12-v2"
# ) # 这个效果太差，相似度：0.15，不选了

# ---------- 2. 创建或获取集合 ----------
# collection 类似于数据库里的表
collection = client.get_or_create_collection(
    name="oa_knowledge_base",
    embedding_function=sentence_transformer_ef
)

# # ---------- 3. 准备数据 ----------
# # 从 documents.py 导入数据
# from documents import oa_tickets, policy_docs

all_docs = oa_tickets + policy_docs

# 提取 ID、内容、元数据
ids = [doc["id"] for doc in all_docs]
documents = [doc["content"] for doc in all_docs]
metadatas = [{"title":doc["title"],"source":"oa_tickets" if doc["id"].startswith("TK") else "policy"} for doc in all_docs]

# ---------- 4. 存入向量库 ----------
# 如果集合不为空，先清空（方便重复运行）
try:
    collection.delete(where={})
except:
    pass

collection.add(ids=ids,documents=documents,metadatas=metadatas)
print(f"成功存入 {len(ids)} 条文档到知识库")
print(f"集合中的文档数量：{collection.count()}")

# ---------- 5. 测试检索 ----------
test_query = "离职需要什么流程？"
results = collection.query(query_texts=[test_query], n_results=3) # 返回最相似的 3 条
print(f"\n查询：{test_query}")
print(f"找到 {len(results['documents'][0])} 条相关文档：")
# print(results)
for i,(doc,metadata,distance) in enumerate(zip(results["documents"][0],results["metadatas"][0],results["distances"][0])):
    print(f"\n--- 结果 {i+1} (相似度：{1-distance:.2f}) ---")
    print(f"标题：{metadata['title']}")
    print(f"内容：{doc[:100]}...")


成功存入 10 条文档到知识库
集合中的文档数量：10

查询：离职需要什么流程？
找到 3 条相关文档：

--- 结果 1 (相似度：0.61) ---
标题：离职流程咨询
内容：员工离职需要在 OA 系统提交离职申请，经部门主管审批、HR 审核、财务清算后，由 HR 办理离职手续。整个流程预计 5-7 个工作日。...

--- 结果 2 (相似度：0.56) ---
标题：加班审批流程
内容：加班需提前在 OA 提交加班申请，注明加班日期、时长、事由。经部门主管审批后生效。紧急情况可在加班后 24 小时内补交申请。...

--- 结果 3 (相似度：0.54) ---
标题：固定资产领用
内容：领用固定资产（电脑、显示器等）需在 OA 提交资产领用申请，经部门主管审批后到行政部领取。贵重设备需 IT 部门登记备案。...


### results = collection.query(query_texts=[test_query], n_results=3)返回的结果如下：

results["documents"][0] ## 回答的内容  
results["metadatas"][0] ## 标题title和数组名称source  
results["distances"][0] ## 距离  

## 第 3 步：写 RAG 查询函数

In [19]:
from dotenv import load_dotenv
import os

import chromadb
from chromadb.utils import embedding_functions
import json
import requests

# ---------- 配置 ----------
load_dotenv()
api_key =os.getenv("OPENROUTER_API_KEY")
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "authorization":f"Bearer {api_key}",
    "content-type":"application/json"
}

# 初始化 Chroma
client = chromadb.PersistentClient(path="./chroma_db")
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.get_collection(name="oa_knowledge_base", embedding_function=sentence_transformer_ef)


def retrieve_documents(query,top_k=3):
    """
    从向量库检索最相关的文档
    
    参数：
        query: string, 用户问题
        top_k: int, 返回的文档数量
    
    返回：
        list, 每个元素包含 title, content, score
    """
    results = collection.query(query_texts=[query], n_results = top_k)
    
    retrieved = []
    for doc,metadata,distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            "title": metadata["title"],
            "content":doc,
            "score": 1 - distance # 距离越小越相似，转为分数
        })
    return retrieved

def rag_query(user_question):
    """
    完整的 RAG 查询：检索 + 生成
    
    参数：
        user_question: string, 用户问题
    
    返回：
        string, 模型回答
    """
    # 第一步：检索相关文档
    relevant_docs= retrieve_documents(user_question,top_k=3)
    # 第二步：构建上下文
    context = "\n\n".join([f"【{doc['title']}】\n{doc['content']}" for doc in relevant_docs])
    # 第三步：构建 prompt
    system_prompt = """你是一个企业知识库助手。请基于以下提供的文档内容回答用户问题。
        要求：
        1. 只基于提供的文档回答，不要编造
        2. 如果文档中没有相关信息，请如实说"文档中没有找到相关信息"
        3. 回答时注明参考来源（文档标题）
        4. 用简洁专业的中文回答
    """
    user_prompt = f"""参考文档：{context}
        用户问题：{user_question}
        请基于以上文档回答：
    """
    # 第四步：调模型
    data = json.dumps({
        "model":"nvidia/nemotron-3.5-lightning:free",
        "messages":[
            {"role":"system", "content":system_prompt},
            {"role":"user", "content":user_prompt}
        ],
        "temperature":0.3 # 降低随机性，让回答更严谨
    })
    resp = requests.post(url=url,headers=headers,data=data)
    resp = resp.json()["choices"][0]["message"]["content"]
    
    return resp, relevant_docs

## 第 4 步：测试 RAG 问答

In [20]:
# ---------- 测试 ----------
if __name__  == "__main__":
    test_questions = [
        "离职需要什么流程？",
        "加班怎么申请？",
        "报销单审批要多久？",
        "BOM 变更要找谁？",
        "年假有多少天？",
        "公司 wifi 密码是多少？",  # 这个文档里没有
    ]
    for q in test_questions:
        print(f"\n{'='*50}")
        print(f"问题：{q}")
        print(f"{'='*50}")
        
        answer, sources = rag_query(q)
        print(f"\n参考文档：")
        for src in sources:
            print(f"  - {src['title']} (相关性：{src['score']:.2f})")
        print(f"\n回答：{answer}")


问题：离职需要什么流程？

参考文档：
  - 离职流程咨询 (相关性：0.61)
  - 加班审批流程 (相关性：0.56)
  - 固定资产领用 (相关性：0.54)

回答：根据【离职流程咨询】，员工离职流程如下：
1. 在OA系统提交离职申请；
2. 经部门主管审批；
3. HR审核；
4. 财务清算；
5. HR办理离职手续。

整个流程预计5-7个工作日。

问题：加班怎么申请？

参考文档：
  - 加班审批流程 (相关性：0.80)
  - 固定资产领用 (相关性：0.54)
  - 离职流程咨询 (相关性：0.52)

回答：根据《加班审批流程》，加班申请需按以下步骤操作：  
- 提前在OA系统提交加班申请，并注明加班日期、时长及事由；  
- 经部门主管审批后方可生效。  
- 紧急情况可在加班后24小时内补交申请。  

参考来源：【加班审批流程】

问题：报销单审批要多久？

参考文档：
  - 考勤管理制度 (相关性：0.57)
  - 加班审批流程 (相关性：0.56)
  - 离职流程咨询 (相关性：0.53)

回答：文档中没有找到相关信息。已检查《考勤管理制度》、《加班审批流程》、《离职流程咨询》三份文档，但未发现关于报销单审批时长的内容。

问题：BOM 变更要找谁？

参考文档：
  - BOM 变更流程 (相关性：0.68)
  - 离职流程咨询 (相关性：0.54)
  - 考勤管理制度 (相关性：0.51)

回答：根据《BOM 变更流程》，BOM 变更流程如下：工程师在 PLM 系统提交变更申请 → 项目经理审批 → 通知物料管理员更新物料清单；涉及产线调整的，同步通知生产计划员。  

因此，若需发起 BOM 变更，请联系工程师提交申请；若需更新物料，联系物料管理员；若涉及产线调整，还需通知生产计划员。  

参考来源：【BOM 变更流程】

问题：年假有多少天？

参考文档：
  - 请假类型及天数 (相关性：0.71)
  - 考勤管理制度 (相关性：0.64)
  - 加班审批流程 (相关性：0.58)

回答：根据《请假类型及天数》文档，年假天数按工龄计算：工龄满1年享5天，满10年享10天。

问题：公司 wifi 密码是多少？

参考文档：
  - 信息安全管理制度 (相关性：0.73)
  - 离职流

## 第 5 步：把 RAG 封装成可复用的模块

In [43]:
import chromadb
from chromadb.utils import embedding_functions
import json
import requests
from dotenv import load_dotenv
import os

class RAGEngine:
    """RAG 引擎，封装检索和生成"""
    
    def __init__(self,collection_name="oa_knowledge_base"):
        load_dotenv()
        self.api_key = os.getenv("OPENROUTER_API_KEY")
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "authorization":f"Bearer {api_key}",
            "content-type":"application/json"
        }
        
        # 初始化chromadb
        self.client = chromadb.PersistentClient(path="./chroma_db")
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.collection = self.client.get_collection(name=collection_name, embedding_function=self.embedding_function)
    
    def retrieve(self, query, top_k=3):
        """检索相关文档"""
        results = self.collection.query(query_texts=[query], n_results=top_k)
        
        retrieved = []
        for doc,metadata,distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
            retrieved.append({
                "title" : metadata["title"],
                "content" : doc,
                "score" : 1-distance
            })
        return retrieved
    
    def query(self, user_question, top_k=3):
        """完整的 RAG 查询"""
        # 检索
        relevant_docs = self.retrieve(user_question,top_k)
        # 构建上下文
        context = "\n\n".join([f"【{doc['title']}】\n{doc['content']}" for doc in relevant_docs])
        
        system_prompt ="""你是一个企业知识库助手，请基于以下提供的文档内容回答用户问题。
        要求：
        1.只基于文档内容回答问题，不要编造
        2.若文档中没有相关信息，请如实说"文档中没有找到相关信息"
        3.回答时著名参考来源(文档标题)
        4.用简洁专业的中文来回答"""
        
        user_prompt =f"""参考文档：{context}
        用户问题：{user_question}
        请基于以上文档回答问题"""
        
        # 调模型
        data = json.dumps({
            "model":"nvidia/nemotron-3.5-lightning:free",
            "messages":[
                {"role":"system","content":system_prompt},
                {"role":"user","content":user_prompt}
            ],
            "temperature":0.3
        })
        
        resp = requests.post(url=self.url, headers=self.headers, data=data)
        resp = resp.json()["choices"][0]["message"]["content"]
        
        return resp, relevant_docs

In [44]:
rag = RAGEngine()
resp, relevant_docs = rag.query("加班怎么申请？")
print(resp)

根据《加班审批流程》，加班申请需**提前在 OA 系统提交**，并注明**加班日期、时长及事由**。审批需经**部门主管批准**后方可生效。若属**紧急情况**，可在加班后**24小时内补交**申请。  
来源：【加班审批流程】
